# Prepare the MatGPT Telco tokenizer locally

This notebook is a thin operator interface over the safe local CLI. It prepares data and tokenizer evidence only; it cannot start model training.


## Goal

Build the representative 200M-token fitting sample, train a separate candidate tokenizer, compare it with the pilot tokenizer, and record an explicit reviewed selection. Run one stage at a time.


## Setup

### Editable settings and source paths


In [ ]:
from pathlib import Path

RUN_STAGE = "tokenizer_sample"  # @param ["tokenizer_sample", "tokenizer_candidate", "tokenizer_compare", "tokenizer_select"]
LOCAL_WORK_ROOT = Path.home() / "matgpt_work" / "matgpt_telco_300m"
DRIVE_PUBLISH_ROOT = Path.home() / "Library/CloudStorage/GoogleDrive-ACCOUNT/My Drive/matgpt_artifacts/matgpt_telco_300m"
REPO_ROOT = Path.cwd()

SOURCES = REPO_ROOT / "configs/data/telco_300m_sources.yaml"
MIXTURE = REPO_ROOT / "configs/data/telco_300m_mixture.yaml"
CANDIDATE_CONFIG = REPO_ROOT / "configs/data/telco_300m_tokenizer_candidate.yaml"
MODEL_CONFIG = REPO_ROOT / "configs/matgpt_telco_300m.yaml"
CONTAMINATION_PATTERN_PATHS = []  # Add every prepared evaluation JSONL path.

SAMPLE_MANIFEST = LOCAL_WORK_ROOT / "tokenizer_sample/manifest.json"
BASELINE_TOKENIZER = DRIVE_PUBLISH_ROOT / "tokenizers/pilot_20m"
CANDIDATE_TOKENIZER = DRIVE_PUBLISH_ROOT / "tokenizers/representative_200m"
HOLDOUT_MANIFEST = SAMPLE_MANIFEST
COMPARISON = DRIVE_PUBLISH_ROOT / "comparison.json"
TOKENIZER_WINNER = ""  # Set only after reviewing comparison.json
APPROVE_SELECTION = False

assert LOCAL_WORK_ROOT.resolve() != DRIVE_PUBLISH_ROOT.resolve()


### Environment and storage evidence

Record this output in the operator log before starting the expensive sample. The target machine has 24GB RAM and should currently have at least 100GiB free locally. The CLI still enforces recipe identity on resume.


In [ ]:
import os
import platform
import shutil
import sys

memory_bytes = None
if hasattr(os, "sysconf"):
    try:
        memory_bytes = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES")
    except (OSError, ValueError):
        pass
local_disk = shutil.disk_usage(Path.home())
print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("RAM GiB:", round(memory_bytes / 1024**3, 1) if memory_bytes else "inspect with Activity Monitor")
print("Local free GiB:", round(local_disk.free / 1024**3, 1))
print("Local work root:", LOCAL_WORK_ROOT)
print("Drive publish root:", DRIVE_PUBLISH_ROOT)


> **Google Drive safety:** In Drive for desktop, keep **My Drive** in **Stream files** mode. Do not mark the `matgpt_artifacts` tree **Available offline**; that can consume the local disk with a second copy. Confirm the real mounted `My Drive` path in Finder before continuing.


## Steps

### 1. Check the selected paths


In [ ]:
assert RUN_STAGE in {"tokenizer_sample", "tokenizer_candidate", "tokenizer_compare", "tokenizer_select"}
assert (REPO_ROOT / "scripts/prepare_telco_local.py").is_file()
for config_path in (SOURCES, MIXTURE, CANDIDATE_CONFIG, MODEL_CONFIG):
    assert config_path.is_file(), f"Missing config: {config_path}"
assert DRIVE_PUBLISH_ROOT.is_dir(), "Edit DRIVE_PUBLISH_ROOT to the existing streamed My Drive directory."
if RUN_STAGE == "tokenizer_sample":
    assert CONTAMINATION_PATTERN_PATHS, "Add every Open Telco Lite/Full JSONL path before sampling."
    assert all(Path(path).is_file() for path in CONTAMINATION_PATTERN_PATHS), "A contamination file is missing."
assert not LOCAL_WORK_ROOT.resolve().is_relative_to(DRIVE_PUBLISH_ROOT.resolve())
assert not DRIVE_PUBLISH_ROOT.resolve().is_relative_to(LOCAL_WORK_ROOT.resolve())
print("Path checks passed for stage:", RUN_STAGE)


### 2. Build and preview the command


In [ ]:
import shlex
import sys

command = [
    sys.executable,
    "scripts/prepare_telco_local.py",
    "--stage", RUN_STAGE,
    "--sources", str(SOURCES),
    "--mixture", str(MIXTURE),
    "--candidate-config", str(CANDIDATE_CONFIG),
    "--model-config", str(MODEL_CONFIG),
    "--work-dir", str(LOCAL_WORK_ROOT),
    "--drive-dir", str(DRIVE_PUBLISH_ROOT),
]
stage_arguments = {
    "tokenizer_sample": [],
    "tokenizer_candidate": ["--sample-manifest", str(SAMPLE_MANIFEST)],
    "tokenizer_compare": [
        "--baseline-tokenizer", str(BASELINE_TOKENIZER),
        "--candidate-tokenizer", str(CANDIDATE_TOKENIZER),
        "--holdout-manifest", str(HOLDOUT_MANIFEST),
    ],
    "tokenizer_select": ["--comparison", str(COMPARISON), "--winner", TOKENIZER_WINNER],
}
if RUN_STAGE == "tokenizer_sample":
    for pattern_path in CONTAMINATION_PATTERN_PATHS:
        command.extend(["--contamination-patterns", str(pattern_path)])
if RUN_STAGE == "tokenizer_select":
    assert TOKENIZER_WINNER, "Review comparison.json and set TOKENIZER_WINNER."
    assert APPROVE_SELECTION, "Set APPROVE_SELECTION=True only after review."
    command.append("--approve")
command.extend(stage_arguments[RUN_STAGE])
print(shlex.join(command))


### 3. Run the selected stage

Output is merged and printed as each line arrives. `Ctrl-C` interrupts the local process; rerun the same sample command to resume from committed units.


In [ ]:
import subprocess

process = subprocess.Popen(
    command,
    cwd=REPO_ROOT,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
assert process.stdout is not None
for line in process.stdout:
    print(line, end="", flush=True)
return_code = process.wait()
if return_code != 0:
    raise RuntimeError(f"Stage {RUN_STAGE} failed with exit code {return_code}.")


## Checks

### Inspect the stage evidence


In [ ]:
result_paths = {
    "tokenizer_sample": SAMPLE_MANIFEST,
    "tokenizer_candidate": CANDIDATE_TOKENIZER / "tokenizer_report.json",
    "tokenizer_compare": COMPARISON,
    "tokenizer_select": DRIVE_PUBLISH_ROOT / "tokenizer_selection.json",
}
result_path = result_paths[RUN_STAGE]
assert result_path.is_file(), f"Expected result is missing: {result_path}"
print("Result:", result_path)
print("Review Drive for the cloud-sync completion icon before relying on published evidence.")


## Next Steps

Run stages in order: `tokenizer_sample` → `tokenizer_candidate` → `tokenizer_compare` → reviewed `tokenizer_select`. Selection never copies over either tokenizer. If `representative_200m` wins, refresh the pilot preparation, smoke, pilot, and evaluation gates under its fingerprint before any full-run approval. This notebook does not provide a model-training action.
